# 02. Construct PheCode + measurement dataset

Adapted from Naomi's `construct_phecode_measurement_dataset.ipynb`.

Key change from the original: instead of selecting one "best" visit per patient,
this notebook uses the precomputed `patient_index.parquet` from notebook 01
(19,858 AD cases + 19,858 non-AD controls, each with a fixed `index_datetime`).

Outputs (in `output/`):
- `04_demographics_features.parquet`
- `05_selected_measurements.parquet`
- `06_measurement_features_long.parquet`
- `07_feature_matrix.parquet`     <- X for notebook 03
- `08_disease_matrix.parquet`     <- Y for notebook 03
- summary CSVs

**Design note on Y:** This is a feature-extraction task, not a prediction task.
AD-related PheCodes are intentionally KEPT in Y, because Y here represents each
patient's phenotype profile (a description of who they are), not a prediction target.
Removing AD labels would discard real information about the AD cohort and would
not meaningfully prevent cohort separability anyway (AD patients differ from
non-AD controls across many comorbidity dimensions, not just AD codes themselves).


## 1. Setup

In [ ]:
# !pip install duckdb pyarrow pandas numpy
from pathlib import Path
import json
import re

import duckdb
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 100)

con = duckdb.connect()
print('DuckDB version:', duckdb.__version__)


## 2. Configuration

In [ ]:
DATA_PATH = '/home/jupyter/2835794-data'
PHECODE_MAP_PATH = 'phecode_icd10.csv'
EXCLUDED_LABELS_PATH = 'final_raw_labels_to_exclude_union_phecode_name_review.txt'
PATIENT_INDEX_PATH = 'output/patient_index.parquet'

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

LOOKBACK_DAYS = 365
MIN_LABEL_PATIENTS = 0
MAX_LABELS = 2000
MIN_MEASUREMENT_PERSON_PERCENT = 0.005
MIN_MEASUREMENT_PATIENTS = None
MAX_MEASUREMENTS = 10000
REQUIRE_MEASUREMENT_IN_LOOKBACK = True

# Measurement concepts to EXCLUDE as features (real target leakage from Naomi's notebook).
# 42538860  Charlson Comorbidity Index -- weighted sum of diagnoses (direct leakage to PheCode Y)
# 37016720  Hospital readmission risk score -- composite of diagnoses
# These are scores COMPUTED FROM diagnoses, not real measurements, so they leak
# regardless of task (prediction or feature extraction). Keep these excluded.
EXCLUDED_MEASUREMENT_CONCEPT_IDS = [42538860, 37016720]
EXCLUDED_CONCEPT_SQL = (
    ' AND m.measurement_concept_id NOT IN ('
    + ','.join(str(x) for x in EXCLUDED_MEASUREMENT_CONCEPT_IDS) + ')'
    if EXCLUDED_MEASUREMENT_CONCEPT_IDS else ''
)


## 3. Helper functions

In [ ]:
def parquet_glob(table):
    return str(Path(DATA_PATH) / table / '*.parquet')

def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

def qident(value):
    return '"' + str(value).replace('"', '""') + '"'

def save_table(table_name, file_name):
    out = OUTPUT_DIR / file_name
    con.execute(f'COPY {table_name} TO {sql_literal(str(out))} (FORMAT PARQUET)')
    print('wrote', out)
    return out

def save_query_csv(query, file_name):
    out = OUTPUT_DIR / file_name
    con.sql(query).to_df().to_csv(out, index=False)
    print('wrote', out)
    return out

person_path = parquet_glob('person')
measurement_path = parquet_glob('measurement')
condition_path = parquet_glob('condition_occurrence')
concept_path = parquet_glob('concept')

print('person:', person_path)
print('measurement:', measurement_path)
print('condition_occurrence:', condition_path)


## 4. Load and normalize ICD-10 to PheCode mapping

In [ ]:
phe = pd.read_csv(PHECODE_MAP_PATH, dtype=str)
phe.columns = [c.strip() for c in phe.columns]

required_cols = {'ALT_CODE', 'ICD10', 'PheCode', 'Phenotype'}
missing = required_cols - set(phe.columns)
if missing:
    raise ValueError(f'Missing expected columns in {PHECODE_MAP_PATH}: {missing}')

def normalize_icd10(x):
    if pd.isna(x):
        return None
    x = str(x).strip().upper()
    x = re.sub(r'[^A-Z0-9]', '', x)
    return x or None

def clean_phecode(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    if not x or x.lower() == 'nan':
        return None
    return x

def format_phecode_label(phecode, phenotype):
    phecode = clean_phecode(phecode)
    if phecode is None:
        return None
    phenotype = '' if pd.isna(phenotype) else str(phenotype).strip()
    phenotype = re.sub(r'[^A-Za-z0-9]', '_', phenotype)
    return f'{phecode}__{phenotype}'

map_rows = []
for _, row in phe.iterrows():
    phecode = clean_phecode(row['PheCode'])
    if phecode is None:
        continue
    for source_col in ['ALT_CODE', 'ICD10']:
        code_norm = normalize_icd10(row[source_col])
        if code_norm:
            map_rows.append({
                'icd10_norm': code_norm,
                'phecode': phecode,
                'phenotype': row['Phenotype'],
                'disease_label': format_phecode_label(phecode, row['Phenotype']),
            })

phe_map = (pd.DataFrame(map_rows)
           .drop_duplicates()
           .dropna(subset=['icd10_norm', 'phecode', 'disease_label']))
con.register('phe_map_df', phe_map)
con.execute('CREATE OR REPLACE TEMP TABLE phecode_map AS SELECT DISTINCT * FROM phe_map_df')

display(phe_map.head())
display(con.sql('''
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT icd10_norm) AS n_icd10_norm,
       COUNT(DISTINCT phecode) AS n_phecodes
FROM phecode_map
''').to_df())


## 5. Load reviewed label exclusions

This loads the curated PheCode exclusion list from `final_raw_labels_to_exclude_union_phecode_name_review.txt`
(Naomi's manual review removing labels that should not be used as outcomes,
e.g. screening/symptom codes). AD PheCodes are NOT excluded — see design note at top.


In [ ]:
exclude_rows = []
exclude_path = Path(EXCLUDED_LABELS_PATH)
if exclude_path.exists():
    for line in exclude_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        phecode = line.split('__', 1)[0].strip()
        exclude_rows.append({'excluded_phecode': phecode, 'excluded_disease_label': line})
else:
    print(f'Exclusion file not found: {exclude_path}. No labels excluded by review list.')

excluded_labels_df = pd.DataFrame(exclude_rows).drop_duplicates()
if excluded_labels_df.empty:
    excluded_labels_df = pd.DataFrame(columns=['excluded_phecode', 'excluded_disease_label'])

con.register('excluded_labels_df', excluded_labels_df)
con.execute('CREATE OR REPLACE TEMP TABLE excluded_labels AS SELECT * FROM excluded_labels_df')
print('Review exclusion labels loaded:', len(excluded_labels_df))
display(excluded_labels_df.head())


## 6. Map condition events to PheCodes (per row, not yet aggregated)

In [ ]:
con.execute(f'''
CREATE OR REPLACE TEMP TABLE condition_phecode AS
SELECT DISTINCT
  c.person_id,
  c.visit_occurrence_id,
  CAST(c.condition_start_datetime AS TIMESTAMP) AS condition_start_datetime,
  CAST(c.condition_start_datetime AS DATE) AS condition_date,
  upper(regexp_replace(trim(c.condition_source_value), '[^A-Za-z0-9]', '', 'g')) AS icd10_norm,
  m.phecode,
  m.phenotype,
  m.disease_label
FROM read_parquet({sql_literal(condition_path)}) c
JOIN phecode_map m
  ON upper(regexp_replace(trim(c.condition_source_value), '[^A-Za-z0-9]', '', 'g')) = m.icd10_norm
WHERE c.condition_start_datetime IS NOT NULL
  AND c.condition_source_value IS NOT NULL
  AND regexp_matches(trim(c.condition_source_value), '^[A-Za-z][0-9]')
''')

con.execute('''
CREATE OR REPLACE TEMP TABLE candidate_condition_labels AS
SELECT cp.*
FROM condition_phecode cp
LEFT JOIN excluded_labels e
  ON cp.phecode = e.excluded_phecode
WHERE e.excluded_phecode IS NULL
  AND cp.phecode IS NOT NULL
  AND trim(cp.phecode) <> ''
  AND lower(trim(cp.phecode)) <> 'nan'
''')

print('All mapped condition events:')
display(con.sql('''
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT person_id) AS n_persons,
       COUNT(DISTINCT phecode) AS n_phecodes
FROM condition_phecode
''').to_df())

print('Candidate condition labels after exclusion:')
display(con.sql('''
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT person_id) AS n_persons,
       COUNT(DISTINCT phecode) AS n_phecodes
FROM candidate_condition_labels
''').to_df())


## 7. Load precomputed `patient_index` (from notebook 01)

This replaces the original notebook's "select best visit per patient" logic.
The patient_index here is fixed: 19,858 AD + 19,858 non-AD.


In [ ]:
# Load patient_index from notebook 01
con.execute(f'''
CREATE OR REPLACE TEMP TABLE patient_index_raw AS
SELECT
  row_id,
  person_id,
  CAST(index_datetime AS TIMESTAMP) AS index_datetime,
  CAST(index_datetime AS DATE) AS index_date,
  CAST(AGE_AT_INDEX AS INTEGER) AS age_at_index,
  group AS cohort_group
FROM read_parquet({sql_literal(PATIENT_INDEX_PATH)})
''')

# Join demographics from person.parquet
con.execute(f'''
CREATE OR REPLACE TEMP TABLE patient_index_with_demo AS
SELECT
  pi.*,
  p.gender_concept_id,
  p.race_concept_id,
  p.ethnicity_concept_id,
  p.gender_source_value,
  p.race_source_value,
  p.ethnicity_source_value,
  p.death_datetime
FROM patient_index_raw pi
LEFT JOIN read_parquet({sql_literal(person_path)}) p USING (person_id)
''')

# Optional filter: require at least one measurement in the lookback window
if REQUIRE_MEASUREMENT_IN_LOOKBACK:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE patient_index AS
    SELECT pi.*
    FROM patient_index_with_demo pi
    WHERE EXISTS (
        SELECT 1
        FROM read_parquet({sql_literal(measurement_path)}) m
        WHERE m.person_id = pi.person_id
          AND m.measurement_datetime >= pi.index_datetime - INTERVAL {int(LOOKBACK_DAYS)} DAY
          AND m.measurement_datetime <= pi.index_datetime
          AND m.value_as_number IS NOT NULL
    )
    ''')
else:
    con.execute('CREATE OR REPLACE TEMP TABLE patient_index AS SELECT * FROM patient_index_with_demo')

print('patient_index after filters:')
display(con.sql('''
SELECT cohort_group, COUNT(*) AS n
FROM patient_index
GROUP BY cohort_group
ORDER BY cohort_group
''').to_df())


## 8. Build labels for each patient (within lookback window before/on index)

In [ ]:
# For each patient, labels are PheCodes from condition events with
# condition_date BETWEEN index_date - LOOKBACK_DAYS AND index_date.
# We use the same lookback window as for measurements, for symmetry.

con.execute(f'''
CREATE OR REPLACE TEMP TABLE labels_long_all AS
SELECT DISTINCT
  pi.row_id,
  pi.person_id,
  c.phecode,
  c.phenotype,
  c.disease_label,
  1 AS label
FROM patient_index pi
JOIN candidate_condition_labels c
  ON pi.person_id = c.person_id
WHERE c.condition_start_datetime >= pi.index_datetime - INTERVAL {int(LOOKBACK_DAYS)} DAY
  AND c.condition_start_datetime <= pi.index_datetime
''')

con.execute(f'''
CREATE OR REPLACE TEMP TABLE selected_labels AS
WITH label_counts AS (
    SELECT
      phecode,
      any_value(phenotype) AS phenotype,
      any_value(disease_label) AS disease_label,
      COUNT(DISTINCT person_id) AS n_positive_persons
    FROM labels_long_all
    WHERE phecode IS NOT NULL
      AND trim(phecode) <> ''
      AND lower(trim(phecode)) <> 'nan'
    GROUP BY phecode
    HAVING COUNT(DISTINCT person_id) >= {int(MIN_LABEL_PATIENTS)}
)
SELECT *
FROM label_counts
ORDER BY n_positive_persons DESC
LIMIT {int(MAX_LABELS)}
''')

con.execute('''
CREATE OR REPLACE TEMP TABLE labels_long AS
SELECT l.*
FROM labels_long_all l
JOIN selected_labels s USING (phecode)
''')

print('Selected labels (top 30 by prevalence):')
display(con.sql('SELECT * FROM selected_labels ORDER BY n_positive_persons DESC LIMIT 30').to_df())
display(con.sql('''
SELECT COUNT(*) AS n_label_rows,
       COUNT(DISTINCT person_id) AS n_positive_persons,
       COUNT(DISTINCT phecode) AS n_labels
FROM labels_long
''').to_df())

save_query_csv('SELECT * FROM selected_labels ORDER BY n_positive_persons DESC',
               '02_selected_labels.csv')


## 9. Demographic features

In [ ]:
con.execute('''
CREATE OR REPLACE TEMP TABLE demographics_features AS
SELECT
  person_id,
  index_datetime,
  cohort_group,
  age_at_index,
  gender_concept_id,
  race_concept_id,
  ethnicity_concept_id,
  gender_source_value,
  race_source_value,
  ethnicity_source_value
FROM patient_index
''')

display(con.sql('SELECT * FROM demographics_features LIMIT 10').to_df())
save_table('demographics_features', '04_demographics_features.parquet')


## 10. Select numeric measurement concepts (prevalence-filtered)

In [ ]:
eligible_person_count = con.sql('SELECT COUNT(DISTINCT person_id) FROM patient_index').fetchone()[0]
if MIN_MEASUREMENT_PATIENTS is None:
    measurement_patient_threshold = max(1, int(np.ceil(eligible_person_count * MIN_MEASUREMENT_PERSON_PERCENT)))
else:
    measurement_patient_threshold = int(MIN_MEASUREMENT_PATIENTS)

print('Indexed persons:', eligible_person_count)
print('Minimum measurement patient percent:', MIN_MEASUREMENT_PERSON_PERCENT)
print('Minimum measurement patient count:', measurement_patient_threshold)

con.execute(f'''
CREATE OR REPLACE TEMP TABLE all_measurement_names AS
SELECT
  m.measurement_concept_id,
  COALESCE(c.concept_name, 'measurement_' || CAST(m.measurement_concept_id AS VARCHAR)) AS measurement_name,
  any_value(m.measurement_source_value) AS measurement_source_value,
  any_value(m.unit_source_value) AS unit_source_value,
  any_value(c.vocabulary_id) AS vocabulary_id,
  any_value(c.concept_code) AS concept_code,
  COUNT(*) AS n_rows,
  COUNT(DISTINCT m.person_id) AS n_persons
FROM read_parquet({sql_literal(measurement_path)}) m
JOIN patient_index pi
  ON m.person_id = pi.person_id
LEFT JOIN read_parquet({sql_literal(concept_path)}) c
  ON m.measurement_concept_id = c.concept_id
WHERE m.measurement_datetime >= pi.index_datetime - INTERVAL {int(LOOKBACK_DAYS)} DAY
  AND m.measurement_datetime <= pi.index_datetime
  AND m.value_as_number IS NOT NULL
  AND m.measurement_concept_id IS NOT NULL
  AND m.measurement_concept_id <> 0{EXCLUDED_CONCEPT_SQL}
GROUP BY m.measurement_concept_id, c.concept_name
''')

display(con.sql('''
SELECT COUNT(*) AS n_all_measurements_before_filter,
       MIN(n_persons) AS min_persons,
       MAX(n_persons) AS max_persons
FROM all_measurement_names
''').to_df())

# Sanitize concept names into safe column names
def sanitize_feature_name(name):
    s = re.sub(r'[^A-Za-z0-9_]', '_', str(name)).strip('_')
    return s.lower() if s else 'unnamed_measurement'

all_names_df = con.sql('SELECT * FROM all_measurement_names').to_df()
all_names_df['feature_name'] = all_names_df['measurement_name'].apply(sanitize_feature_name)

# Disambiguate any name collisions
seen = {}
final_names = []
for name in all_names_df['feature_name']:
    if name in seen:
        seen[name] += 1
        final_names.append(f'{name}_{seen[name]}')
    else:
        seen[name] = 0
        final_names.append(name)
all_names_df['feature_name'] = final_names

selected_measurement_features = (
    all_names_df[all_names_df['n_persons'] >= measurement_patient_threshold]
    .sort_values(['n_persons', 'n_rows'], ascending=[False, False])
    .head(MAX_MEASUREMENTS)
    .reset_index(drop=True)
)

con.register('selected_measurement_features_df', selected_measurement_features)
con.execute('''
CREATE OR REPLACE TEMP TABLE selected_measurement_features AS
SELECT * FROM selected_measurement_features_df
''')

print('Selected measurement features:', len(selected_measurement_features))
display(selected_measurement_features.head(20))
save_table('selected_measurement_features', '05_selected_measurements.parquet')


## 11. Extract latest measurement values per (person, measurement) within lookback

In [ ]:
con.execute(f'''
CREATE OR REPLACE TEMP TABLE measurement_features_long AS
WITH ranked_measurements AS (
    SELECT
      pi.row_id,
      pi.person_id,
      pi.index_datetime,
      m.measurement_concept_id,
      s.feature_name,
      m.value_as_number,
      m.unit_source_value,
      m.measurement_datetime,
      ROW_NUMBER() OVER (
        PARTITION BY pi.person_id, m.measurement_concept_id
        ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
      ) AS rn
    FROM patient_index pi
    JOIN read_parquet({sql_literal(measurement_path)}) m
      ON pi.person_id = m.person_id
    JOIN selected_measurement_features s
      ON m.measurement_concept_id = s.measurement_concept_id
    WHERE m.measurement_datetime >= pi.index_datetime - INTERVAL {int(LOOKBACK_DAYS)} DAY
      AND m.measurement_datetime <= pi.index_datetime
      AND m.value_as_number IS NOT NULL
      AND m.measurement_concept_id <> 0{EXCLUDED_CONCEPT_SQL}
)
SELECT
  row_id,
  person_id,
  index_datetime,
  measurement_concept_id,
  feature_name,
  value_as_number,
  unit_source_value,
  measurement_datetime
FROM ranked_measurements
WHERE rn = 1
''')

print('Each row is the latest value per (patient, measurement) within lookback.')
display(con.sql('SELECT * FROM measurement_features_long LIMIT 10').to_df())
display(con.sql('''
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT person_id) AS n_persons,
       COUNT(DISTINCT measurement_concept_id) AS n_measurements
FROM measurement_features_long
''').to_df())
save_table('measurement_features_long', '06_measurement_features_long.parquet')


## 12. Pivot to wide feature matrix and join demographics

In [ ]:
feature_metadata = con.sql('''
SELECT measurement_concept_id, feature_name
FROM selected_measurement_features
ORDER BY n_persons DESC, n_rows DESC
''').fetchall()

feature_exprs = []
for concept_id, feature_name in feature_metadata:
    feature_exprs.append(
        f'MAX(CASE WHEN f.measurement_concept_id = {int(concept_id)} '
        f'THEN f.value_as_number END) AS {qident(feature_name)}'
    )

feature_pivot_sql = ',\n  '.join(feature_exprs) if feature_exprs else 'NULL AS no_measurement_features'

con.execute(f'''
CREATE OR REPLACE TEMP TABLE measurement_features_wide AS
SELECT
  person_id,
  {feature_pivot_sql}
FROM measurement_features_long f
GROUP BY person_id
''')

con.execute('''
CREATE OR REPLACE TEMP TABLE feature_matrix AS
SELECT
  pi.row_id,
  d.person_id,
  d.index_datetime,
  d.cohort_group,
  d.age_at_index,
  d.gender_source_value,
  d.race_source_value,
  d.ethnicity_source_value,
  mw.* EXCLUDE (person_id)
FROM patient_index pi
JOIN demographics_features d USING (person_id)
LEFT JOIN measurement_features_wide mw USING (person_id)
ORDER BY pi.row_id
''')

print('feature_matrix: one row per patient, demographics + one column per measurement')
display(con.sql('SELECT COUNT(*) AS n_rows FROM feature_matrix').to_df())
display(con.sql('SELECT * FROM feature_matrix LIMIT 5').to_df())
save_table('feature_matrix', '07_feature_matrix.parquet')


## 13. Build disease matrix (Y)

In [ ]:
selected_label_rows = con.sql('''
SELECT phecode, disease_label
FROM selected_labels
ORDER BY n_positive_persons DESC
''').fetchall()
selected_label_names = [str(x[1]) for x in selected_label_rows]

label_exprs = []
for phecode, disease_label in selected_label_rows:
    label_exprs.append(
        f'MAX(CASE WHEN l.phecode = {sql_literal(phecode)} THEN 1 ELSE 0 END) '
        f'AS {qident(disease_label)}'
    )

label_pivot_sql = ',\n  '.join(label_exprs) if label_exprs else '0 AS no_selected_labels'

con.execute(f'''
CREATE OR REPLACE TEMP TABLE disease_matrix AS
SELECT
  pi.row_id,
  pi.person_id,
  {label_pivot_sql}
FROM patient_index pi
LEFT JOIN labels_long l
  ON pi.person_id = l.person_id
 AND pi.row_id = l.row_id
GROUP BY pi.row_id, pi.person_id
ORDER BY pi.row_id
''')

disease_cols = con.sql('DESCRIBE SELECT * FROM disease_matrix').to_df()['column_name'].tolist()
if 'nan' in disease_cols:
    con.execute(f'ALTER TABLE disease_matrix DROP COLUMN {qident("nan")}')
    print('Dropped invalid label column: nan')

print('disease_matrix: one row per patient, one column per PheCode (named phecode__disease_name)')
display(con.sql('SELECT COUNT(*) AS n_rows FROM disease_matrix').to_df())
display(con.sql('SELECT * FROM disease_matrix LIMIT 5').to_df())
save_table('disease_matrix', '08_disease_matrix.parquet')
save_query_csv('''
SELECT phecode, phenotype AS disease_name, disease_label, n_positive_persons
FROM selected_labels
WHERE lower(trim(phecode)) <> 'nan'
ORDER BY n_positive_persons DESC
''', 'disease_names.csv')


## 14. Alignment check

In [ ]:
alignment_check = con.sql('''
SELECT
  COUNT(*) AS n_rows_checked,
  SUM(CASE WHEN f.row_id = d.row_id AND f.person_id = d.person_id THEN 1 ELSE 0 END) AS n_aligned,
  SUM(CASE WHEN f.row_id <> d.row_id OR f.person_id <> d.person_id THEN 1 ELSE 0 END) AS n_misaligned
FROM feature_matrix f
JOIN disease_matrix d ON f.row_id = d.row_id
''').to_df()
display(alignment_check)


## 15. Summary

In [ ]:
n_rows = con.sql('SELECT COUNT(*) FROM feature_matrix').fetchone()[0]
n_disease_rows = con.sql('SELECT COUNT(*) FROM disease_matrix').fetchone()[0]
n_feature_cols = len(con.sql('DESCRIBE SELECT * FROM feature_matrix').to_df())
n_label_cols = len(selected_label_rows)

per_group = con.sql('''
SELECT cohort_group, COUNT(*) AS n
FROM feature_matrix
GROUP BY cohort_group
''').to_df()

# Check whether AD-related PheCodes are present in Y (they SHOULD be, per design note)
ad_phecodes_check = con.sql('''
SELECT phecode, disease_label, n_positive_persons
FROM selected_labels
WHERE phecode LIKE '290%'
ORDER BY phecode
''').to_df()

summary = pd.DataFrame([{
    'data_path': DATA_PATH,
    'patient_index_path': PATIENT_INDEX_PATH,
    'lookback_days': LOOKBACK_DAYS,
    'require_measurement_in_lookback': REQUIRE_MEASUREMENT_IN_LOOKBACK,
    'n_feature_rows': n_rows,
    'n_disease_rows': n_disease_rows,
    'n_feature_columns_including_ids': n_feature_cols,
    'n_label_columns': n_label_cols,
    'min_measurement_person_percent': MIN_MEASUREMENT_PERSON_PERCENT,
    'min_measurement_patients': measurement_patient_threshold,
    'ad_phecodes_retained_in_Y': True,
    'n_ad_related_phecode_columns_in_Y': len(ad_phecodes_check),
}])
summary.to_csv(OUTPUT_DIR / '10_dataset_summary.csv', index=False)

print('Per-group counts in final feature_matrix:')
display(per_group)
print()
print('AD-related PheCodes present in Y (retained intentionally, see design note):')
display(ad_phecodes_check)
print()
display(summary)


## 16. Output manifest

In [ ]:
for path in sorted(OUTPUT_DIR.glob('*')):
    print(path)
